In [3]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
pdf_dir = "../data/pdf"
loader = DirectoryLoader(
    pdf_dir,
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
    )
docs = loader.load()

In [6]:
docs

[Document(metadata={'producer': 'Pdftools SDK', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-04-25T03:19:30+00:00', 'source': '..\\data\\pdf\\report2024.pdf', 'total_pages': 133, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'Pdftools SDK', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-04-25T03:19:30+00:00', 'source': '..\\data\\pdf\\report2024.pdf', 'total_pages': 133, 'page': 1, 'page_label': '2'}, page_content='EXPONENTIAL THINKING – SUSTAINABLE BUILDING  |      3\nMỤC LỤC\nCHÚNG TÔI LÀ AI 4\nTẦM NHÌN CỦA TECH 6\nThông điệp của Chủ tịch Hội đồng quản trị  8\nThông điệp của Tổng giám đốc 10\nCÂU CHUYỆN CỦA TECH 18\nVề chúng tôi 20\nTầm nhìn và Sứ mệnh 22\nCâu chuyện Thương hiệu 24\nChặng đường Lịch sử 26\nCơ cấu Cổ đông và Hoạt động Quan hệ Nhà đầu tư 28\nTHÀNH TỰU CỦA TECH 34\nBáo cáo Toàn cảnh Ngân hàng 36\nKhối Ngân hàng Bán lẻ (RBG) 44\nKhối Ngân hàng Doanh nghiệp và Định chế Tài chính (CIBG) 50\nKhối Ngân hàng Giao dịch T

In [8]:
len(docs)

133

In [9]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain.embeddings.base import Embeddings
from sentence_transformers import SentenceTransformer

In [12]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of documents objects,return a new list of document objects
    containing only 'source' in the metadata and original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [13]:
minimal_docs = filter_to_minimal_docs(docs)

In [14]:
minimal_docs

[Document(metadata={'source': '..\\data\\pdf\\report2024.pdf'}, page_content=''),
 Document(metadata={'source': '..\\data\\pdf\\report2024.pdf'}, page_content='EXPONENTIAL THINKING – SUSTAINABLE BUILDING  |      3\nMỤC LỤC\nCHÚNG TÔI LÀ AI 4\nTẦM NHÌN CỦA TECH 6\nThông điệp của Chủ tịch Hội đồng quản trị  8\nThông điệp của Tổng giám đốc 10\nCÂU CHUYỆN CỦA TECH 18\nVề chúng tôi 20\nTầm nhìn và Sứ mệnh 22\nCâu chuyện Thương hiệu 24\nChặng đường Lịch sử 26\nCơ cấu Cổ đông và Hoạt động Quan hệ Nhà đầu tư 28\nTHÀNH TỰU CỦA TECH 34\nBáo cáo Toàn cảnh Ngân hàng 36\nKhối Ngân hàng Bán lẻ (RBG) 44\nKhối Ngân hàng Doanh nghiệp và Định chế Tài chính (CIBG) 50\nKhối Ngân hàng Giao dịch Toàn cầu (GTS) 56\nCác Công ty con 60\nCHUYỂN ĐỔI SỐ TẠI TECH 70\nBáo cáo Chuyển đổi Ngân hàng 72\nKhối Dữ liệu và Phân tích (DnA) 76\nVăn phòng Chuyển đổi số (DO + IT) 80\nKhối Quản trị Nguồn nhân lực (HR) 90\nPHÁT TRIỂN BỀN VỮNG CÙNG TECH 94\nQuản trị Doanh nghiệp 96\nQuản trị Rủi ro 156\nVăn hoá Doanh nghiệp 160\

In [ ]:
# text splitter
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size= 1000,
        chunk_overlap = 200
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [16]:
texts_chunk = text_split(minimal_docs)
print(f"texts_chunk: {len(texts_chunk)}")

texts_chunk: 882


In [17]:
texts_chunk

[Document(metadata={'source': '..\\data\\pdf\\report2024.pdf'}, page_content='EXPONENTIAL THINKING – SUSTAINABLE BUILDING  |      3\nMỤC LỤC\nCHÚNG TÔI LÀ AI 4\nTẦM NHÌN CỦA TECH 6\nThông điệp của Chủ tịch Hội đồng quản trị  8\nThông điệp của Tổng giám đốc 10\nCÂU CHUYỆN CỦA TECH 18\nVề chúng tôi 20\nTầm nhìn và Sứ mệnh 22\nCâu chuyện Thương hiệu 24\nChặng đường Lịch sử 26\nCơ cấu Cổ đông và Hoạt động Quan hệ Nhà đầu tư 28\nTHÀNH TỰU CỦA TECH 34\nBáo cáo Toàn cảnh Ngân hàng 36\nKhối Ngân hàng Bán lẻ (RBG) 44\nKhối Ngân hàng Doanh nghiệp và Định chế Tài chính (CIBG) 50\nKhối Ngân hàng Giao dịch Toàn cầu (GTS) 56\nCác Công ty con 60\nCHUYỂN ĐỔI SỐ TẠI TECH 70\nBáo cáo Chuyển đổi Ngân hàng 72\nKhối Dữ liệu và Phân tích (DnA) 76\nVăn phòng Chuyển đổi số (DO + IT) 80\nKhối Quản trị Nguồn nhân lực (HR) 90\nPHÁT TRIỂN BỀN VỮNG CÙNG TECH 94\nQuản trị Doanh nghiệp 96\nQuản trị Rủi ro 156\nVăn hoá Doanh nghiệp 160\nBáo cáo Phát triển Bền vững 164\nTHÀNH CÔNG NỐI TIẾP CỦA TECH 190\nBáo cáo tài ch

In [10]:
# ── Custom Embedding ────────────────────────────────
_model = SentenceTransformer("AITeamVN/Vietnamese_Embedding_v2")

class CustomEmbedding(Embeddings):
    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return _model.encode(texts, show_progress_bar=True).tolist()

    def embed_query(self, text: str) -> list[float]:
        return _model.encode([text])[0].tolist()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [19]:
embedding = CustomEmbedding()

In [20]:
vector = embedding.embed_query("xin chào, bạn có thể hỗ trợ tôi một số thông tin về sản phẩm không?")
vector

[-0.025990985333919525,
 -0.009741210378706455,
 -0.06707059592008591,
 -0.007820763625204563,
 -0.010081810876727104,
 -0.10255138576030731,
 -0.03681549057364464,
 0.06522330641746521,
 -0.03202627971768379,
 0.030275413766503334,
 0.002839880995452404,
 0.01608528383076191,
 -0.04318824037909508,
 -0.02857884019613266,
 -0.004299099091440439,
 -0.0028225127607584,
 -0.018592925742268562,
 -0.03696153685450554,
 -0.004733589477837086,
 -0.0395558699965477,
 -0.02604975923895836,
 0.034321196377277374,
 0.021654125303030014,
 0.02884058468043804,
 0.043882548809051514,
 -0.0008961104904301465,
 -0.030619898810982704,
 0.0051491232588887215,
 0.04967718943953514,
 -0.04593188688158989,
 0.029064161702990532,
 0.015911653637886047,
 0.018707742914557457,
 0.014090264216065407,
 -0.0031438032165169716,
 -0.04744867607951164,
 -0.003320524003356695,
 -0.02553815022110939,
 -0.030910175293684006,
 0.007409734185785055,
 -0.01075324323028326,
 -0.022657431662082672,
 0.026326080784201622,
 

In [21]:
len(vector)

1024

In [22]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [23]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

In [24]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [25]:
pc

In [26]:
from pinecone import ServerlessSpec

index_name = "techcombank-rag"

if not pc.has_index(index_name):
    pc.create_index(
        index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        )
    )

index = pc.Index(index_name)

In [27]:
docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name,
)

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# load Existing index

from langchain_pinecone import PineconeVectorStore
# embed each chunk and upsert the embeddings into your Pinecone index
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding,
)

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [ ]:
retriever_docs = retriever.invoke("Cho tôi biết trong điều kiện bảo hành, Sản phẩm phải đáp ứng các điều kiện gì?")
retriever_docs